**Purpose**:
 - Train supplier-risk classification models
 - Use temporal validation
 - Compare candidate algorithms
 - Track experiments with MLflow
 - Evaluate final model on unseen 2024 data
 - Retrain production candidate using all labeled history
 - Score 2026 suppliers

**Development design**
-  2022 + 2023 -> grouped stratified cross-validation by SupplierID
-  2024 -> final temporal test
-  2026 -> current scoring population


**Imports & Configuration**

In [0]:
import math
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import sklearn
import mlflow
import mlflow.sklearn

from mlflow.models import infer_signature

from sklearn.base import clone

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.linear_model import (
    LogisticRegression
)

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.model_selection import (
    StratifiedGroupKFold
)

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    accuracy_score,
    confusion_matrix,
    precision_recall_curve
)

from sklearn.utils.class_weight import (
    compute_sample_weight
)

from pyspark.sql import functions as F


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_STATE = 20260802


# ------------------------------------------------------------
# Temporal model design
# ------------------------------------------------------------

TRAIN_YEAR = 2022

VALIDATION_YEAR = 2023

DEVELOPMENT_YEARS = [
    TRAIN_YEAR,
    VALIDATION_YEAR
]

TEST_YEAR = 2024

SCORING_YEAR = 2026


# ------------------------------------------------------------
# Grouped CV configuration
# ------------------------------------------------------------

CV_FOLDS = 5


# ------------------------------------------------------------
# Business / model evaluation configuration
# ------------------------------------------------------------

TARGET_COLUMN = "HighRiskNextYearFlag"

BRD_PRECISION_TARGET = 0.85

F_BETA = 2.0


print(
    "DB_03 configuration loaded."
)

print(
    "scikit-learn version:",
    sklearn.__version__
)

print(
    "MLflow version:",
    mlflow.__version__
)

print(
    "Development years:",
    DEVELOPMENT_YEARS
)

print(
    "Grouped CV folds:",
    CV_FOLDS
)

print(
    "Untouched temporal test year:",
    TEST_YEAR
)

print(
    "Scoring year:",
    SCORING_YEAR
)

DB_03 configuration loaded.
scikit-learn version: 1.7.2
MLflow version: 3.12.0
Development years: [2022, 2023]
Grouped CV folds: 5
Untouched temporal test year: 2024
Scoring year: 2026


**Load OneLake credentials**

In [0]:
# ============================================================
# Load Fabric OneLake credentials securely
# ============================================================

tenant_id = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-tenant-id"
)

client_id = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-client-id"
)

client_secret = dbutils.secrets.get(
    scope="fabric-onelake",
    key="fabric-client-secret"
)


print(
    "Fabric OneLake credentials loaded securely."
)

Fabric OneLake credentials loaded securely.


**Configure OneLake OAuth**

In [0]:
# ============================================================
# Configure Fabric OneLake OAuth
# ============================================================

spark.conf.set(
    "fs.azure.account.auth.type",
    "OAuth"
)

spark.conf.set(
    "fs.azure.account.oauth.provider.type",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.id",
    client_id
)

spark.conf.set(
    "fs.azure.account.oauth2.client.secret",
    client_secret
)

spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint",
    (
        f"https://login.microsoftonline.com/"
        f"{tenant_id}/oauth2/token"
    )
)


print(
    "OneLake OAuth configuration applied."
)

OneLake OAuth configuration applied.


**Define DB_02 input and DB_03 output paths**

abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Files/ml/supplier_risk/training_features

In [0]:
# ============================================================
# ML feature and model output paths
# ============================================================

TRAINING_FEATURES_PATH = (
    "abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Files/ml/supplier_risk/training_features"
)


# ------------------------------------------------------------
# Derive the Supplier Risk ML root
# ------------------------------------------------------------

SUPPLIER_RISK_ML_ROOT = (
    TRAINING_FEATURES_PATH
    .rsplit(
        "/training_features",
        1
    )[0]
)


SCORING_FEATURES_PATH = (
    f"{SUPPLIER_RISK_ML_ROOT}/"
    f"scoring_features"
)


PREDICTIONS_PATH = (
    f"{SUPPLIER_RISK_ML_ROOT}/"
    f"predictions_2026"
)


MODEL_METADATA_PATH = (
    f"{SUPPLIER_RISK_ML_ROOT}/"
    f"model_metadata"
)


FEATURE_IMPORTANCE_PATH = (
    f"{SUPPLIER_RISK_ML_ROOT}/"
    f"feature_importance"
)


print("Training:")
print(TRAINING_FEATURES_PATH)

print("\nScoring:")
print(SCORING_FEATURES_PATH)

print("\nPredictions:")
print(PREDICTIONS_PATH)

print("\nModel metadata:")
print(MODEL_METADATA_PATH)

Training:
abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Files/ml/supplier_risk/training_features

Scoring:
abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Files/ml/supplier_risk/scoring_features

Predictions:
abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Files/ml/supplier_risk/predictions_2026

Model metadata:
abfss://Project_Procurement@onelake.dfs.fabric.microsoft.com/lh_procurement_gold.Lakehouse/Files/ml/supplier_risk/model_metadata


**Read DB_02 feature datasets**

In [0]:
# ============================================================
# Read DB_02 persisted feature datasets
# ============================================================

training_features_spark_df = (
    spark.read
    .format("delta")
    .load(
        TRAINING_FEATURES_PATH
    )
)


scoring_features_spark_df = (
    spark.read
    .format("delta")
    .load(
        SCORING_FEATURES_PATH
    )
)


training_row_count = (
    training_features_spark_df.count()
)

scoring_row_count = (
    scoring_features_spark_df.count()
)


print(
    "Training rows:",
    f"{training_row_count:,}"
)

print(
    "Scoring rows:",
    f"{scoring_row_count:,}"
)

Training rows: 1,009
Scoring rows: 356


**Define model features**

This cell deliberately excludes non predictors:

- SupplierKey
- SupplierID
- SupplierName
- FeatureYear
- FutureOutcomeYear
- SourceAsOfDate
- FeatureEngineeringTimestampUTC


In [0]:
# ============================================================
# Define Supplier Risk model features
# ============================================================

categorical_features = [
    "SupplierType",
    "Country",
    "Region",
    "ESGRating",
    "SupplierStatus"
]


numeric_base_features = [
    # --------------------------------------------------------
    # Supplier attributes
    # --------------------------------------------------------

    "PreferredSupplierFlag",
    "StrategicSupplierFlag",
    "FinancialRiskScore",

    # --------------------------------------------------------
    # Compliance / operational risk
    # --------------------------------------------------------

    "ContractCompliancePct",
    "MaverickSpendPct",

    "SupplierOTDPct",
    "LateFullyReceivedPct",
    "OverdueOpenDeliveryExposurePct",

    "SupplierQualityIndexPct",
    "InvoiceDisputePct",
    "DuplicateInvoicePct",
    "ThreeWayMatchPct",
    "InvoiceExceptionPct",

    # --------------------------------------------------------
    # Supplier activity / exposure
    # --------------------------------------------------------

    "AnnualizedEligibleSpendEUR",
    "AnnualizedFullyReceivedPOItemCount",
    "AnnualizedInvoiceCount",
    "AnnualizedDisputedInvoiceCount",
    "AnnualizedOverdueOpenDeliveryCount",

    # --------------------------------------------------------
    # Historical behavior
    # --------------------------------------------------------

    "PriorYearSpendEUR",
    "YoYSpendChangePct",

    "PriorYearOTDPct",
    "YoYOTDChangePp",

    "PriorYearInvoiceDisputePct",
    "YoYInvoiceDisputeChangePp",

    "PriorYearMaverickSpendPct",
    "YoYMaverickSpendChangePp",

    # --------------------------------------------------------
    # Spend stability / importance
    # --------------------------------------------------------

    "Rolling3YObservationCount",
    "Rolling3YAverageSpendEUR",
    "Rolling3YSpendStdDevEUR",
    "SpendVolatilityPct",

    "SupplierSpendSharePct",
    "LogAnnualizedEligibleSpendEUR",

    # --------------------------------------------------------
    # Data observation quality
    # --------------------------------------------------------

    "RiskMetricObservedCount",
    "RiskMetricCoveragePct"
]


# ------------------------------------------------------------
# Missing-history indicators
#
# Missing historical information can itself carry information.
# We retain that signal instead of only imputing the value.
# ------------------------------------------------------------

history_flag_map = {
    "PriorYearSpendEUR":
        "HasPriorYearSpendFlag",

    "PriorYearOTDPct":
        "HasPriorYearOTDFlag",

    "PriorYearInvoiceDisputePct":
        "HasPriorYearInvoiceDisputeFlag",

    "PriorYearMaverickSpendPct":
        "HasPriorYearMaverickSpendFlag",

    "SpendVolatilityPct":
        "HasSpendVolatilityHistoryFlag"
}


history_flag_features = list(
    history_flag_map.values()
)


numeric_features = (
    numeric_base_features
    + history_flag_features
)


model_features = (
    numeric_features
    + categorical_features
)


print(
    "Numeric features:",
    len(numeric_features)
)

print(
    "Categorical features:",
    len(categorical_features)
)

print(
    "Total raw model features:",
    len(model_features)
)

Numeric features: 39
Categorical features: 5
Total raw model features: 44


**Validate persisted DB_02 schema**

In [0]:
# ============================================================
# Validate DB_02 persisted feature schema
# ============================================================

required_training_columns = (
    [
        "SupplierKey",
        "SupplierID",
        "SupplierName",
        "FeatureYear",
        "FutureOutcomeYear",
        TARGET_COLUMN,
        "SourceAsOfDate"
    ]
    + numeric_base_features
    + categorical_features
)


required_scoring_columns = (
    [
        "SupplierKey",
        "SupplierID",
        "SupplierName",
        "FeatureYear",
        "SourceAsOfDate"
    ]
    + numeric_base_features
    + categorical_features
)


missing_training_columns = sorted(
    set(required_training_columns)
    - set(training_features_spark_df.columns)
)


missing_scoring_columns = sorted(
    set(required_scoring_columns)
    - set(scoring_features_spark_df.columns)
)


if missing_training_columns:
    raise ValueError(
        "Missing training columns: "
        + ", ".join(
            missing_training_columns
        )
    )


if missing_scoring_columns:
    raise ValueError(
        "Missing scoring columns: "
        + ", ".join(
            missing_scoring_columns
        )
    )


print(
    "DB_02 feature schema validation PASSED."
)

DB_02 feature schema validation PASSED.


**Convert the small ML datasets to pandas and add history flags**

With 1.006 rows as a training set, single-node scikit-learn is the appropriate model execution pattern.

In [0]:
# ============================================================
# Convert feature datasets to pandas,
# add history availability indicators,
# and normalize model dtypes
# ============================================================

training_pd = (
    training_features_spark_df
    .select(
        *required_training_columns
    )
    .toPandas()
)


scoring_pd = (
    scoring_features_spark_df
    .select(
        *required_scoring_columns
    )
    .toPandas()
)


# ------------------------------------------------------------
# Add historical-data availability indicators
# ------------------------------------------------------------

def add_history_availability_flags(
    dataframe
):

    df = dataframe.copy()


    for (
        source_feature,
        indicator_feature
    ) in history_flag_map.items():

        df[
            indicator_feature
        ] = (
            df[
                source_feature
            ]
            .notna()
            .astype(
                "float64"
            )
        )


    return df


training_pd = (
    add_history_availability_flags(
        training_pd
    )
)


scoring_pd = (
    add_history_availability_flags(
        scoring_pd
    )
)


# ------------------------------------------------------------
# Normalize every numeric predictor to float64
#
# This prevents MLflow schema problems where integer columns
# cannot later accommodate missing values.
# ------------------------------------------------------------

for column_name in numeric_base_features:

    training_pd[
        column_name
    ] = (
        pd.to_numeric(
            training_pd[
                column_name
            ],
            errors="coerce"
        )
        .astype(
            "float64"
        )
    )


    scoring_pd[
        column_name
    ] = (
        pd.to_numeric(
            scoring_pd[
                column_name
            ],
            errors="coerce"
        )
        .astype(
            "float64"
        )
    )


for column_name in history_flag_features:

    training_pd[
        column_name
    ] = (
        training_pd[
            column_name
        ]
        .astype(
            "float64"
        )
    )


    scoring_pd[
        column_name
    ] = (
        scoring_pd[
            column_name
        ]
        .astype(
            "float64"
        )
    )


# ------------------------------------------------------------
# Target must remain binary integer
# ------------------------------------------------------------

training_pd[
    TARGET_COLUMN
] = (
    training_pd[
        TARGET_COLUMN
    ]
    .astype(
        "int64"
    )
)


print(
    "Training pandas rows:",
    f"{len(training_pd):,}"
)

print(
    "Scoring pandas rows:",
    f"{len(scoring_pd):,}"
)

print(
    "History availability flags added:",
    history_flag_features
)

print(
    "Numeric model features normalized to float64."
)

Training pandas rows: 1,009
Scoring pandas rows: 356
History availability flags added: ['HasPriorYearSpendFlag', 'HasPriorYearOTDFlag', 'HasPriorYearInvoiceDisputeFlag', 'HasPriorYearMaverickSpendFlag', 'HasSpendVolatilityHistoryFlag']
Numeric model features normalized to float64.


**Create temporal development splits**

In [0]:
# ============================================================
# Create development and final temporal-test datasets
# ============================================================

development_pd = (
    training_pd[
        training_pd[
            "FeatureYear"
        ].isin(
            DEVELOPMENT_YEARS
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


test_pd = (
    training_pd[
        training_pd[
            "FeatureYear"
        ]
        == TEST_YEAR
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Development population
# ------------------------------------------------------------

X_development = (
    development_pd[
        model_features
    ]
    .copy()
)


y_development = (
    development_pd[
        TARGET_COLUMN
    ]
    .astype(int)
)


development_groups = (
    development_pd[
        "SupplierID"
    ]
    .astype(str)
)


# ------------------------------------------------------------
# Untouched 2024 temporal test
# ------------------------------------------------------------

X_test = (
    test_pd[
        model_features
    ]
    .copy()
)


y_test = (
    test_pd[
        TARGET_COLUMN
    ]
    .astype(int)
)


# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

def print_split_summary(
    split_name,
    target_series
):

    total = len(
        target_series
    )

    positives = int(
        target_series.sum()
    )

    positive_rate = (
        positives
        / total
        if total > 0
        else 0.0
    )


    print(
        f"{split_name}: "
        f"{total:,} rows | "
        f"{positives:,} high risk | "
        f"{positive_rate:.2%}"
    )


for year_value in DEVELOPMENT_YEARS:

    year_target = (
        development_pd.loc[
            development_pd[
                "FeatureYear"
            ]
            == year_value,
            TARGET_COLUMN
        ]
    )


    print_split_summary(
        f"DEVELOPMENT {year_value}",
        year_target
    )


print_split_summary(
    "DEVELOPMENT TOTAL",
    y_development
)


print_split_summary(
    f"TEMPORAL TEST {TEST_YEAR}",
    y_test
)


# ------------------------------------------------------------
# Quality validation
# ------------------------------------------------------------

if (
    y_development.nunique()
    != 2
):

    raise ValueError(
        "Development population does not "
        "contain both target classes."
    )


if (
    y_test.nunique()
    != 2
):

    raise ValueError(
        "Temporal test population does not "
        "contain both target classes."
    )


development_supplier_count = (
    development_groups
    .nunique()
)


if (
    development_supplier_count
    < CV_FOLDS
):

    raise ValueError(
        "Insufficient supplier groups "
        "for grouped cross-validation."
    )


print(
    "\nDevelopment suppliers:",
    development_supplier_count
)

print(
    "Temporal split validation PASSED."
)

DEVELOPMENT 2022: 326 rows | 92 high risk | 28.22%
DEVELOPMENT 2023: 338 rows | 93 high risk | 27.51%
DEVELOPMENT TOTAL: 664 rows | 185 high risk | 27.86%
TEMPORAL TEST 2024: 345 rows | 96 high risk | 27.83%

Development suppliers: 363
Temporal split validation PASSED.


**Build preprocessing pipeline**

- Numeric nulls use median imputation.
- Categorical nulls use Unknown.
- Categorical variables such as ESG rating remain categorical and are one-hot encoded.

In [0]:
# ============================================================
# Build preprocessing pipeline
# ============================================================

def build_preprocessor():

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    keep_empty_features=True
                )
            ),

            (
                "scaler",
                StandardScaler()
            )
        ]
    )


    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="Unknown",
                    keep_empty_features=True
                )
            ),

            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]
    )


    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_features
            ),

            (
                "categorical",
                categorical_pipeline,
                categorical_features
            )
        ],

        remainder="drop",

        verbose_feature_names_out=False
    )


    return preprocessor


print(
    "Preprocessing pipeline configured."
)

print(
    "Numeric missing values:"
    " median imputation"
)

print(
    "Categorical missing values:"
    " Unknown"
)

print(
    "Empty feature columns:"
    " retained"
)

Preprocessing pipeline configured.
Numeric missing values: median imputation
Categorical missing values: Unknown
Empty feature columns: retained


**Define candidate models**

Model compared:

- Logistic Regression
- Random Forest
- Gradient Boosting

In [0]:
# ============================================================
# Define candidate supplier-risk models
# ============================================================

candidate_specs = {

    "LogisticRegression": {
        "estimator":
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                solver="liblinear",
                random_state=RANDOM_STATE
            ),

        "use_sample_weight":
            False
    },


    "RandomForest": {
        "estimator":
            RandomForestClassifier(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=5,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1
            ),

        "use_sample_weight":
            False
    },


    "GradientBoosting": {
        "estimator":
            GradientBoostingClassifier(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=2,
                min_samples_leaf=8,
                subsample=0.80,
                random_state=RANDOM_STATE
            ),

        "use_sample_weight":
            True
    }
}


def build_model_pipeline(
    model_name
):

    estimator = clone(
        candidate_specs[
            model_name
        ][
            "estimator"
        ]
    )


    return Pipeline(
        steps=[
            (
                "preprocessor",
                build_preprocessor()
            ),
            (
                "model",
                estimator
            )
        ]
    )


def fit_model_pipeline(
    model_name,
    X,
    y
):

    pipeline = (
        build_model_pipeline(
            model_name
        )
    )


    if candidate_specs[
        model_name
    ][
        "use_sample_weight"
    ]:

        sample_weights = (
            compute_sample_weight(
                class_weight="balanced",
                y=y
            )
        )

        pipeline.fit(
            X,
            y,
            model__sample_weight=sample_weights
        )

    else:

        pipeline.fit(
            X,
            y
        )


    return pipeline


print(
    "Candidate models configured:",
    list(
        candidate_specs.keys()
    )
)

Candidate models configured: ['LogisticRegression', 'RandomForest', 'GradientBoosting']


**Define evaluation and threshold functions**

The model is selected using validation PR-AUC.

The operating threshold is separately optimized using F2, giving recall more weight than precision.

In [0]:
# ============================================================
# Classification evaluation and threshold helpers
# ============================================================

def calculate_classification_metrics(
    y_true,
    probabilities,
    threshold
):

    predictions = (
        probabilities
        >= threshold
    ).astype(int)


    return {
        "roc_auc":
            float(
                roc_auc_score(
                    y_true,
                    probabilities
                )
            ),

        "pr_auc":
            float(
                average_precision_score(
                    y_true,
                    probabilities
                )
            ),

        "precision":
            float(
                precision_score(
                    y_true,
                    predictions,
                    zero_division=0
                )
            ),

        "recall":
            float(
                recall_score(
                    y_true,
                    predictions,
                    zero_division=0
                )
            ),

        "f1":
            float(
                f1_score(
                    y_true,
                    predictions,
                    zero_division=0
                )
            ),

        # Keep F2 as a reporting metric.
        # It is no longer used to choose the threshold.
        "f2":
            float(
                fbeta_score(
                    y_true,
                    predictions,
                    beta=F_BETA,
                    zero_division=0
                )
            ),

        "accuracy":
            float(
                accuracy_score(
                    y_true,
                    predictions
                )
            )
    }


# ============================================================
# F1-optimized operating threshold
#
# Why:
# F1 balances precision and recall equally.
#
# The old F2 threshold placed substantially more weight on
# recall, which produced an extremely low threshold and
# caused almost every supplier to be predicted high risk.
#
# We also reject thresholds that predict only one class.
# ============================================================

def find_f1_optimal_threshold(
    y_true,
    probabilities
):

    y_array = np.asarray(
        y_true
    ).astype(int)


    probability_array = np.asarray(
        probabilities,
        dtype=float
    )


    candidate_thresholds = (
        np.unique(
            probability_array
        )
    )


    best_result = None


    for threshold in candidate_thresholds:

        predictions = (
            probability_array
            >= threshold
        ).astype(int)


        # ----------------------------------------------------
        # Reject completely degenerate thresholds
        # ----------------------------------------------------

        if (
            np.unique(
                predictions
            ).size
            < 2
        ):
            continue


        precision_value = float(
            precision_score(
                y_array,
                predictions,
                zero_division=0
            )
        )


        recall_value = float(
            recall_score(
                y_array,
                predictions,
                zero_division=0
            )
        )


        f1_value = float(
            f1_score(
                y_array,
                predictions,
                zero_division=0
            )
        )


        # ----------------------------------------------------
        # Primary objective: highest F1
        #
        # Tie breakers:
        # 1. higher precision
        # 2. higher recall
        # 3. higher threshold
        # ----------------------------------------------------

        result = (
            f1_value,
            precision_value,
            recall_value,
            float(
                threshold
            )
        )


        if (
            best_result is None
            or
            result > best_result
        ):
            best_result = result


    if best_result is None:

        raise ValueError(
            "No non-degenerate F1 threshold could "
            "be identified from development OOF predictions."
        )


    best_f1 = (
        best_result[0]
    )


    best_precision = (
        best_result[1]
    )


    best_recall = (
        best_result[2]
    )


    best_threshold = (
        best_result[3]
    )


    print(
        "F1-optimized development threshold:",
        round(
            best_threshold,
            4
        )
    )

    print(
        "Development precision at threshold:",
        round(
            best_precision,
            4
        )
    )

    print(
        "Development recall at threshold:",
        round(
            best_recall,
            4
        )
    )

    print(
        "Development F1 at threshold:",
        round(
            best_f1,
            4
        )
    )


    return float(
        best_threshold
    )


# ============================================================
# Business precision-target diagnostic
#
# This remains diagnostic only.
# We do NOT force the model threshold to meet 85% precision.
# ============================================================

def best_recall_at_precision_target(
    y_true,
    probabilities,
    precision_target
):

    (
        precision_values,
        recall_values,
        thresholds
    ) = precision_recall_curve(
        y_true,
        probabilities
    )


    candidates = []


    for index, threshold in enumerate(
        thresholds
    ):

        precision_value = float(
            precision_values[
                index
            ]
        )

        recall_value = float(
            recall_values[
                index
            ]
        )


        if (
            precision_value
            >= precision_target
        ):

            candidates.append(
                (
                    recall_value,
                    precision_value,
                    float(
                        threshold
                    )
                )
            )


    if not candidates:

        return None


    candidates.sort(
        reverse=True
    )


    return candidates[0]


print(
    "Evaluation functions loaded."
)

print(
    "Decision-threshold objective: F1"
)

print(
    "F2 retained as reporting metric only."
)

Evaluation functions loaded.
Decision-threshold objective: F1
F2 retained as reporting metric only.


**MLflow model logging helper**

This supports both MLflow 2 and MLflow 3 style model logging so the notebook is less sensitive to the exact runtime version.

In [0]:
# ============================================================
# MLflow compatibility helper
# ============================================================

MLFLOW_MAJOR_VERSION = int(
    mlflow.__version__
    .split(".")[0]
)


def log_sklearn_model_compat(
    model,
    X_example,
    model_name="model"
):

    example = (
        X_example
        .head(10)
        .copy()
    )


    example_predictions = (
        model.predict_proba(
            example
        )[:, 1]
    )


    signature = infer_signature(
        example,
        example_predictions
    )


    common_arguments = {
        "sk_model":
            model,

        "signature":
            signature,

        "input_example":
            example,

        "serialization_format":
            "cloudpickle"
    }


    if MLFLOW_MAJOR_VERSION >= 3:

        return mlflow.sklearn.log_model(
            name=model_name,
            **common_arguments
        )


    return mlflow.sklearn.log_model(
        artifact_path=model_name,
        **common_arguments
    )


print(
    "MLflow major version:",
    MLFLOW_MAJOR_VERSION
)

print(
    "MLflow logging helper ready."
)

MLflow major version: 3
MLflow logging helper ready.


**Train and validate all candidate models**

In [0]:
# ============================================================
# Grouped cross-validation model development
#
# Development population:
# 2022 + 2023
#
# Group:
# SupplierID
#
# Model selection:
# OOF PR-AUC
#
# Threshold selection:
# OOF F1
#
# 2024 remains completely untouched.
# ============================================================

if (
    mlflow.active_run()
    is not None
):

    mlflow.end_run()


candidate_development_models = {}

candidate_thresholds = {}

candidate_model_uris = {}

candidate_oof_probabilities = {}

candidate_results = []


# ------------------------------------------------------------
# Cross-validation definition
# ------------------------------------------------------------

grouped_cv = (
    StratifiedGroupKFold(
        n_splits=CV_FOLDS,
        shuffle=True,
        random_state=RANDOM_STATE
    )
)


# ============================================================
# Candidate-model loop
# ============================================================

for model_name in candidate_specs.keys():

    print(
        f"\nDeveloping {model_name}..."
    )


    # --------------------------------------------------------
    # Out-of-fold predictions
    # --------------------------------------------------------

    oof_probability = np.full(
        shape=len(
            X_development
        ),
        fill_value=np.nan,
        dtype=float
    )


    fold_pr_auc_values = []

    fold_roc_auc_values = []


    # --------------------------------------------------------
    # Grouped cross-validation
    # --------------------------------------------------------

    for (
        fold_number,
        (
            fold_train_index,
            fold_validation_index
        )
    ) in enumerate(

        grouped_cv.split(
            X_development,
            y_development,
            groups=development_groups
        ),

        start=1
    ):

        X_fold_train = (
            X_development

            .iloc[
                fold_train_index
            ]

            .copy()
        )


        y_fold_train = (
            y_development

            .iloc[
                fold_train_index
            ]

            .copy()
        )


        X_fold_validation = (
            X_development

            .iloc[
                fold_validation_index
            ]

            .copy()
        )


        y_fold_validation = (
            y_development

            .iloc[
                fold_validation_index
            ]

            .copy()
        )


        # ----------------------------------------------------
        # Verify supplier-group isolation
        # ----------------------------------------------------

        train_supplier_groups = set(
            development_groups

            .iloc[
                fold_train_index
            ]
        )


        validation_supplier_groups = set(
            development_groups

            .iloc[
                fold_validation_index
            ]
        )


        supplier_overlap = (
            train_supplier_groups

            .intersection(
                validation_supplier_groups
            )
        )


        if supplier_overlap:

            raise ValueError(
                f"Supplier leakage detected "
                f"in fold {fold_number}."
            )


        # ----------------------------------------------------
        # Fit fold model
        # ----------------------------------------------------

        fold_pipeline = (
            fit_model_pipeline(
                model_name,
                X_fold_train,
                y_fold_train
            )
        )


        fold_probability = (
            fold_pipeline

            .predict_proba(
                X_fold_validation
            )[:, 1]
        )


        oof_probability[
            fold_validation_index
        ] = fold_probability


        # ----------------------------------------------------
        # Fold ranking diagnostics
        # ----------------------------------------------------

        fold_pr_auc = float(
            average_precision_score(
                y_fold_validation,
                fold_probability
            )
        )


        fold_roc_auc = float(
            roc_auc_score(
                y_fold_validation,
                fold_probability
            )
        )


        fold_pr_auc_values.append(
            fold_pr_auc
        )


        fold_roc_auc_values.append(
            fold_roc_auc
        )


        print(
            f"  Fold {fold_number}: "
            f"PR-AUC={fold_pr_auc:.4f} | "
            f"ROC-AUC={fold_roc_auc:.4f}"
        )


    # --------------------------------------------------------
    # Validate OOF prediction completeness
    # --------------------------------------------------------

    if np.isnan(
        oof_probability
    ).any():

        raise ValueError(
            f"OOF prediction generation "
            f"failed for {model_name}."
        )


    # --------------------------------------------------------
    # Select operating threshold from DEVELOPMENT only
    #
    # F1 is used rather than F2.
    # --------------------------------------------------------

    optimal_threshold = (
        find_f1_optimal_threshold(
            y_development,
            oof_probability
        )
    )


    # --------------------------------------------------------
    # Default 0.50 metrics for comparison
    # --------------------------------------------------------

    default_metrics = (
        calculate_classification_metrics(
            y_development,
            oof_probability,
            threshold=0.50
        )
    )


    # --------------------------------------------------------
    # F1-optimized operating metrics
    # --------------------------------------------------------

    tuned_metrics = (
        calculate_classification_metrics(
            y_development,
            oof_probability,
            threshold=optimal_threshold
        )
    )


    # --------------------------------------------------------
    # BRD precision target remains diagnostic
    # --------------------------------------------------------

    precision_target_result = (
        best_recall_at_precision_target(
            y_development,
            oof_probability,
            BRD_PRECISION_TARGET
        )
    )


    # --------------------------------------------------------
    # Fit candidate model on full development population
    #
    # 2024 is still not used here.
    # --------------------------------------------------------

    development_pipeline = (
        fit_model_pipeline(
            model_name,
            X_development,
            y_development
        )
    )


    # --------------------------------------------------------
    # MLflow
    # --------------------------------------------------------

    with mlflow.start_run(
        run_name=(
            f"supplier_risk_"
            f"{model_name}_"
            f"grouped_cv"
        )
    ) as run:

        mlflow.set_tags({
            "project":
                (
                    "Enterprise Procurement "
                    "Intelligence Platform"
                ),

            "model_family":
                "Supplier Risk",

            "model_name":
                model_name,

            "model_stage":
                "grouped_cross_validation",

            "development_years":
                "2022,2023",

            "grouping_variable":
                "SupplierID",

            "temporal_test_year":
                str(
                    TEST_YEAR
                )
        })


        mlflow.log_params({
            "random_state":
                RANDOM_STATE,

            "cv_folds":
                CV_FOLDS,

            "development_years":
                "2022,2023",

            "grouping_variable":
                "SupplierID",

            "decision_threshold":
                optimal_threshold,

            "threshold_objective":
                "F1",

            "business_precision_target":
                BRD_PRECISION_TARGET
        })


        mlflow.log_metrics({
            "development_oof_roc_auc":
                tuned_metrics[
                    "roc_auc"
                ],

            "development_oof_pr_auc":
                tuned_metrics[
                    "pr_auc"
                ],

            "development_oof_precision":
                tuned_metrics[
                    "precision"
                ],

            "development_oof_recall":
                tuned_metrics[
                    "recall"
                ],

            "development_oof_f1":
                tuned_metrics[
                    "f1"
                ],

            "development_oof_f2":
                tuned_metrics[
                    "f2"
                ],

            "mean_fold_pr_auc":
                float(
                    np.mean(
                        fold_pr_auc_values
                    )
                ),

            "std_fold_pr_auc":
                float(
                    np.std(
                        fold_pr_auc_values
                    )
                ),

            "mean_fold_roc_auc":
                float(
                    np.mean(
                        fold_roc_auc_values
                    )
                ),

            "std_fold_roc_auc":
                float(
                    np.std(
                        fold_roc_auc_values
                    )
                ),

            "default_threshold_precision":
                default_metrics[
                    "precision"
                ],

            "default_threshold_recall":
                default_metrics[
                    "recall"
                ]
        })


        model_info = (
            log_sklearn_model_compat(
                development_pipeline,
                X_development,
                model_name="model"
            )
        )


        run_id = (
            run.info.run_id
        )


    # --------------------------------------------------------
    # Retain candidate outputs
    # --------------------------------------------------------

    candidate_development_models[
        model_name
    ] = development_pipeline


    candidate_thresholds[
        model_name
    ] = optimal_threshold


    candidate_model_uris[
        model_name
    ] = model_info.model_uri


    candidate_oof_probabilities[
        model_name
    ] = oof_probability


    # --------------------------------------------------------
    # Candidate-model summary
    # --------------------------------------------------------

    result_row = {

        "ModelName":
            model_name,

        "DevelopmentOOFROCAUC":
            tuned_metrics[
                "roc_auc"
            ],

        "DevelopmentOOFPRAUC":
            tuned_metrics[
                "pr_auc"
            ],

        "MeanFoldPRAUC":
            float(
                np.mean(
                    fold_pr_auc_values
                )
            ),

        "StdFoldPRAUC":
            float(
                np.std(
                    fold_pr_auc_values
                )
            ),

        "DecisionThreshold":
            optimal_threshold,

        "DevelopmentOOFPrecision":
            tuned_metrics[
                "precision"
            ],

        "DevelopmentOOFRecall":
            tuned_metrics[
                "recall"
            ],

        "DevelopmentOOFF1":
            tuned_metrics[
                "f1"
            ],

        "DevelopmentOOFF2":
            tuned_metrics[
                "f2"
            ],

        "MLflowRunID":
            run_id,

        "ModelURI":
            model_info.model_uri
    }


    if (
        precision_target_result
        is not None
    ):

        (
            target_recall,
            target_precision,
            target_threshold
        ) = (
            precision_target_result
        )


        result_row[
            "BestRecallAt85PctPrecision"
        ] = target_recall


        result_row[
            "PrecisionAtTargetThreshold"
        ] = target_precision


        result_row[
            "ThresholdAt85PctPrecision"
        ] = target_threshold


    else:

        result_row[
            "BestRecallAt85PctPrecision"
        ] = np.nan


        result_row[
            "PrecisionAtTargetThreshold"
        ] = np.nan


        result_row[
            "ThresholdAt85PctPrecision"
        ] = np.nan


    candidate_results.append(
        result_row
    )


print(
    "\nGrouped cross-validation "
    "candidate development completed."
)

print(
    "Threshold objective: F1"
)

print(
    "2024 has not been used "
    "for model or threshold selection."
)


Developing LogisticRegression...
  Fold 1: PR-AUC=0.3865 | ROC-AUC=0.5920
  Fold 2: PR-AUC=0.3088 | ROC-AUC=0.5743
  Fold 3: PR-AUC=0.3560 | ROC-AUC=0.5994
  Fold 4: PR-AUC=0.5236 | ROC-AUC=0.6075
  Fold 5: PR-AUC=0.4430 | ROC-AUC=0.6289
F1-optimized development threshold: 0.1338
Development precision at threshold: 0.2883
Development recall at threshold: 0.9568
Development F1 at threshold: 0.4431


🔗 View Logged Model at: https://adb-7405606393632123.3.azuredatabricks.net/ml/experiments/445824903590900/models/m-5e3e876d02a240899a1108cd36d1fb43?o=7405606393632123



Developing RandomForest...
  Fold 1: PR-AUC=0.4918 | ROC-AUC=0.6883
  Fold 2: PR-AUC=0.3263 | ROC-AUC=0.6726
  Fold 3: PR-AUC=0.3556 | ROC-AUC=0.6027
  Fold 4: PR-AUC=0.5203 | ROC-AUC=0.6696
  Fold 5: PR-AUC=0.6156 | ROC-AUC=0.7392
F1-optimized development threshold: 0.418
Development precision at threshold: 0.404
Development recall at threshold: 0.6595
Development F1 at threshold: 0.501


🔗 View Logged Model at: https://adb-7405606393632123.3.azuredatabricks.net/ml/experiments/445824903590900/models/m-5c17ff91cbfb4f84b1813f29ca40da78?o=7405606393632123



Developing GradientBoosting...
  Fold 1: PR-AUC=0.4538 | ROC-AUC=0.6794
  Fold 2: PR-AUC=0.3364 | ROC-AUC=0.6305
  Fold 3: PR-AUC=0.3516 | ROC-AUC=0.6318
  Fold 4: PR-AUC=0.4987 | ROC-AUC=0.6608
  Fold 5: PR-AUC=0.5105 | ROC-AUC=0.6822
F1-optimized development threshold: 0.3987
Development precision at threshold: 0.3832
Development recall at threshold: 0.6919
Development F1 at threshold: 0.4933


🔗 View Logged Model at: https://adb-7405606393632123.3.azuredatabricks.net/ml/experiments/445824903590900/models/m-80d02fc2bd4244b9a2a4eefd1de66fe0?o=7405606393632123



Grouped cross-validation candidate development completed.
Threshold objective: F1
2024 has not been used for model or threshold selection.


**Compare candidate models and select the winner**

The winner is selected by Validation PR-AUC.

In [0]:
# ============================================================
# Compare candidate models and select winner
#
# Primary selection criterion:
# Development OOF PR-AUC
#
# 2024 remains untouched.
# ============================================================

candidate_results_pd = (
    pd.DataFrame(
        candidate_results
    )

    .sort_values(
        by=[
            "DevelopmentOOFPRAUC",
            "DevelopmentOOFROCAUC",
            "DevelopmentOOFRecall"
        ],

        ascending=[
            False,
            False,
            False
        ]
    )

    .reset_index(
        drop=True
    )
)


display(
    candidate_results_pd
)


BEST_MODEL_NAME = (
    candidate_results_pd
    .iloc[0][
        "ModelName"
    ]
)


SELECTED_THRESHOLD = float(
    candidate_results_pd
    .iloc[0][
        "DecisionThreshold"
    ]
)


print(
    "Selected model:",
    BEST_MODEL_NAME
)

print(
    "Selected OOF decision threshold:",
    round(
        SELECTED_THRESHOLD,
        4
    )
)

print(
    "Selection criterion:",
    "highest development OOF PR-AUC"
)

print(
    "Development years:",
    DEVELOPMENT_YEARS
)

print(
    "Supplier grouping:",
    "SupplierID"
)

print(
    "Untouched temporal test:",
    TEST_YEAR
)

ModelName,DevelopmentOOFROCAUC,DevelopmentOOFPRAUC,MeanFoldPRAUC,StdFoldPRAUC,DecisionThreshold,DevelopmentOOFPrecision,DevelopmentOOFRecall,DevelopmentOOFF1,DevelopmentOOFF2,MLflowRunID,ModelURI,BestRecallAt85PctPrecision,PrecisionAtTargetThreshold,ThresholdAt85PctPrecision
RandomForest,0.6677537662923886,0.4265984961623047,0.4619133728624186,0.10735931514805404,0.4179641151487885,0.40397350993377484,0.6594594594594595,0.5010266940451745,0.5854126679462572,e2097db71d5f444c8bfedfe25662802d,models:/m-5c17ff91cbfb4f84b1813f29ca40da78,null,null,null
GradientBoosting,0.6503413643288383,0.4073203963101192,0.4301950879979703,0.07302057641861824,0.3987415548999144,0.38323353293413176,0.6918918918918919,0.4932562620423892,0.595903165735568,3d0e84a15384463fa9769cddff2aa83f,models:/m-80d02fc2bd4244b9a2a4eefd1de66fe0,null,null,null
LogisticRegression,0.594526885967387,0.39420767756655595,0.40356519631594867,0.07416554479894034,0.1337846631227048,0.28827361563517917,0.9567567567567568,0.4430538172715895,0.6536189069423929,f4001ea9551e49b3a1d6a3e3a163a163,models:/m-5e3e876d02a240899a1108cd36d1fb43,0.016216216216216217,1.0,0.9757859975158304


Selected model: RandomForest
Selected OOF decision threshold: 0.418
Selection criterion: highest development OOF PR-AUC
Development years: [2022, 2023]
Supplier grouping: SupplierID
Untouched temporal test: 2024


**Refit selected model on 2022 + 2023 and evaluate 2024**

In [0]:
# ============================================================
# Final temporal evaluation
#
# Development:
# 2022 + 2023
#
# Model + threshold already selected using grouped OOF CV.
#
# Final untouched temporal holdout:
# 2024
# ============================================================

temporal_test_pipeline = (
    fit_model_pipeline(
        BEST_MODEL_NAME,
        X_development,
        y_development
    )
)


# ------------------------------------------------------------
# Score untouched 2024 population
# ------------------------------------------------------------

test_probability = (
    temporal_test_pipeline
    .predict_proba(
        X_test
    )[:, 1]
)


test_metrics = (
    calculate_classification_metrics(
        y_test,
        test_probability,
        threshold=SELECTED_THRESHOLD
    )
)


test_predictions = (
    test_probability
    >= SELECTED_THRESHOLD
).astype(int)


test_confusion_matrix = (
    confusion_matrix(
        y_test,
        test_predictions
    )
)


print(
    "Final temporal test completed."
)

print(
    "Development years:",
    DEVELOPMENT_YEARS
)

print(
    "Development rows:",
    f"{len(X_development):,}"
)

print(
    "Temporal test year:",
    TEST_YEAR
)

print(
    "Temporal test rows:",
    f"{len(X_test):,}"
)

print(
    "Selected model:",
    BEST_MODEL_NAME
)

print(
    "Selected OOF threshold:",
    round(
        SELECTED_THRESHOLD,
        4
    )
)

Final temporal test completed.
Development years: [2022, 2023]
Development rows: 664
Temporal test year: 2024
Temporal test rows: 345
Selected model: RandomForest
Selected OOF threshold: 0.418


**Display final temporal-test performance**

In [0]:
# ============================================================
# Display final temporal test metrics
# ============================================================

test_metrics_pd = pd.DataFrame(
    [
        {
            "ModelName":
                BEST_MODEL_NAME,

            "DecisionThreshold":
                SELECTED_THRESHOLD,

            "TestROCAUC":
                test_metrics[
                    "roc_auc"
                ],

            "TestPRAUC":
                test_metrics[
                    "pr_auc"
                ],

            "TestPrecision":
                test_metrics[
                    "precision"
                ],

            "TestRecall":
                test_metrics[
                    "recall"
                ],

            "TestF1":
                test_metrics[
                    "f1"
                ],

            "TestF2":
                test_metrics[
                    "f2"
                ],

            "TestAccuracy":
                test_metrics[
                    "accuracy"
                ],

            "BRDPrecisionTarget":
                BRD_PRECISION_TARGET,

            "BRDPrecisionTargetMetFlag":
                int(
                    test_metrics[
                        "precision"
                    ]
                    >= BRD_PRECISION_TARGET
                )
        }
    ]
)


display(
    test_metrics_pd
)

ModelName,DecisionThreshold,TestROCAUC,TestPRAUC,TestPrecision,TestRecall,TestF1,TestF2,TestAccuracy,BRDPrecisionTarget,BRDPrecisionTargetMetFlag
RandomForest,0.4179641151487885,0.5494059571619814,0.35876460738766247,0.3006535947712418,0.4791666666666667,0.36947791164658633,0.42830540037243947,0.5449275362318841,0.85,0


**Display confusion matrix**

In [0]:
# ============================================================
# Temporal test confusion matrix
# ============================================================

tn, fp, fn, tp = (
    test_confusion_matrix
    .ravel()
)


confusion_matrix_pd = pd.DataFrame(
    [
        {
            "ActualClass":
                "Low Risk",

            "PredictedLowRisk":
                int(tn),

            "PredictedHighRisk":
                int(fp)
        },

        {
            "ActualClass":
                "High Risk",

            "PredictedLowRisk":
                int(fn),

            "PredictedHighRisk":
                int(tp)
        }
    ]
)


display(
    confusion_matrix_pd
)


print(
    "True negatives:",
    int(tn)
)

print(
    "False positives:",
    int(fp)
)

print(
    "False negatives:",
    int(fn)
)

print(
    "True positives:",
    int(tp)
)

ActualClass,PredictedLowRisk,PredictedHighRisk
Low Risk,142,107
High Risk,50,46


True negatives: 142
False positives: 107
False negatives: 50
True positives: 46


In [0]:
# ============================================================
# Diagnose univariate predictive signal
#
# Purpose:
# Determine whether individual supplier-risk features contain
# meaningful information about next-year HighRiskNextYearFlag.
# ============================================================

from sklearn.metrics import roc_auc_score


diagnostic_rows = []


for feature_name in numeric_base_features:

    feature_series = (
        development_pd[
            feature_name
        ]
        .copy()
    )


    valid_mask = (
        feature_series.notna()
        &
        development_pd[
            TARGET_COLUMN
        ].notna()
    )


    x = (
        feature_series[
            valid_mask
        ]
    )


    y = (
        development_pd.loc[
            valid_mask,
            TARGET_COLUMN
        ]
        .astype(int)
    )


    if (
        len(x) == 0
        or
        x.nunique() < 2
        or
        y.nunique() < 2
    ):
        continue


    raw_auc = roc_auc_score(
        y,
        x
    )


    # Direction-independent discrimination.
    # Example:
    # low OTD may imply risk, so AUC could naturally be < 0.5.
    discrimination_auc = max(
        raw_auc,
        1.0 - raw_auc
    )


    high_risk_mean = (
        development_pd.loc[
            valid_mask
            &
            (
                development_pd[
                    TARGET_COLUMN
                ] == 1
            ),
            feature_name
        ]
        .mean()
    )


    low_risk_mean = (
        development_pd.loc[
            valid_mask
            &
            (
                development_pd[
                    TARGET_COLUMN
                ] == 0
            ),
            feature_name
        ]
        .mean()
    )


    diagnostic_rows.append(
        {
            "FeatureName":
                feature_name,

            "ObservedRows":
                int(
                    valid_mask.sum()
                ),

            "LowRiskMean":
                float(
                    low_risk_mean
                ),

            "HighRiskMean":
                float(
                    high_risk_mean
                ),

            "RawAUC":
                float(
                    raw_auc
                ),

            "DirectionIndependentAUC":
                float(
                    discrimination_auc
                )
        }
    )


univariate_signal_pd = (
    pd.DataFrame(
        diagnostic_rows
    )

    .sort_values(
        "DirectionIndependentAUC",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


display(
    univariate_signal_pd.head(25)
)

FeatureName,ObservedRows,LowRiskMean,HighRiskMean,RawAUC,DirectionIndependentAUC
AnnualizedFullyReceivedPOItemCount,664,24.931106471816285,39.005405405405405,0.6408170174349715,0.6408170174349715
AnnualizedInvoiceCount,664,6.450939457202505,9.805405405405406,0.6293404051232862,0.6293404051232862
AnnualizedEligibleSpendEUR,664,2472594.2281210856,6343324.63945946,0.6266659143485865,0.6266659143485865
LogAnnualizedEligibleSpendEUR,664,12.802608855800976,13.68234326982809,0.6266659143485865,0.6266659143485865
SupplierSpendSharePct,664,0.2090710975466797,0.5270551979649333,0.6246797946171642,0.6246797946171642
Rolling3YAverageSpendEUR,664,2543240.8688935284,7316937.0890270285,0.6162162162162163,0.6162162162162163
FinancialRiskScore,664,31.204592901878915,39.832432432432434,0.6009874174801106,0.6009874174801106
Rolling3YSpendStdDevEUR,304,1304849.1865040518,4641395.832859883,0.6000847502516022,0.6000847502516022
MaverickSpendPct,656,76.73004228329809,66.53267759562841,0.4036899686918749,0.5963100313081251
ContractCompliancePct,656,23.269957716701903,33.467322404371586,0.5963100313081251,0.5963100313081251


**Log the final temporal-test model**

In [0]:
# ============================================================
# Log final temporal-test model to MLflow
# ============================================================

if mlflow.active_run() is not None:

    mlflow.end_run()


with mlflow.start_run(
    run_name=(
        f"supplier_risk_"
        f"{BEST_MODEL_NAME}_"
        f"temporal_test"
    )
) as temporal_run:

    mlflow.set_tags({
        "project":
            "Enterprise Procurement Intelligence Platform",

        "model_family":
            "Supplier Risk",

        "selected_model":
            BEST_MODEL_NAME,

        "model_stage":
            "temporal_test",

        "development_years":
            f"{TRAIN_YEAR}-{VALIDATION_YEAR}",

        "test_year":
            str(
                TEST_YEAR
            )
    })


    mlflow.log_params({
        "decision_threshold":
            SELECTED_THRESHOLD,

        "threshold_objective":
            "F1",

        "business_precision_target":
            BRD_PRECISION_TARGET
    })


    mlflow.log_metrics({
        "test_roc_auc":
            test_metrics[
                "roc_auc"
            ],

        "test_pr_auc":
            test_metrics[
                "pr_auc"
            ],

        "test_precision":
            test_metrics[
                "precision"
            ],

        "test_recall":
            test_metrics[
                "recall"
            ],

        "test_f1":
            test_metrics[
                "f1"
            ],

        "test_f2":
            test_metrics[
                "f2"
            ],

        "test_accuracy":
            test_metrics[
                "accuracy"
            ],

        "test_false_positive_count":
            int(
                fp
            ),

        "test_false_negative_count":
            int(
                fn
            ),

        "test_true_positive_count":
            int(
                tp
            ),

        "test_true_negative_count":
            int(
                tn
            )
    })


    temporal_test_model_info = (
        log_sklearn_model_compat(
            temporal_test_pipeline,
            X_development,
            model_name="model"
        )
    )


    TEMPORAL_TEST_RUN_ID = (
        temporal_run.info.run_id
    )


    TEMPORAL_TEST_MODEL_URI = (
        temporal_test_model_info.model_uri
    )


print(
    "Temporal test model logged."
)

print(
    "Run ID:",
    TEMPORAL_TEST_RUN_ID
)

print(
    "Model URI:",
    TEMPORAL_TEST_MODEL_URI
)

print(
    "Threshold objective: F1"
)

🔗 View Logged Model at: https://adb-7405606393632123.3.azuredatabricks.net/ml/experiments/445824903590900/models/m-d69f503b9a02488e946192630829bdae?o=7405606393632123


Temporal test model logged.
Run ID: f4c6e5c81e314faab58f77047823940b
Model URI: models:/m-d69f503b9a02488e946192630829bdae
Threshold objective: F1


**Inspect model feature importance**

Adapt automatically depending which candidate won.

In [0]:
# ============================================================
# Inspect model feature importance
# ============================================================

fitted_preprocessor = (
    temporal_test_pipeline
    .named_steps[
        "preprocessor"
    ]
)


fitted_estimator = (
    temporal_test_pipeline
    .named_steps[
        "model"
    ]
)


encoded_feature_names = (
    fitted_preprocessor
    .get_feature_names_out()
)


if hasattr(
    fitted_estimator,
    "coef_"
):

    raw_values = (
        fitted_estimator
        .coef_[0]
    )


    feature_importance_pd = pd.DataFrame({
        "FeatureName":
            encoded_feature_names,

        "Importance":
            np.abs(
                raw_values
            ),

        "Direction":
            raw_values
    })


elif hasattr(
    fitted_estimator,
    "feature_importances_"
):

    raw_values = (
        fitted_estimator
        .feature_importances_
    )


    feature_importance_pd = pd.DataFrame({
        "FeatureName":
            encoded_feature_names,

        "Importance":
            raw_values,

        "Direction":
            np.nan
    })


else:

    raise ValueError(
        "Selected estimator does not expose "
        "feature importance information."
    )


feature_importance_pd = (
    feature_importance_pd

    .sort_values(
        "Importance",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


feature_importance_pd[
    "Rank"
] = (
    np.arange(
        1,
        len(
            feature_importance_pd
        ) + 1
    )
)


display(
    feature_importance_pd
    .head(25)
)

FeatureName,Importance,Direction,Rank
FinancialRiskScore,0.06051661218150884,null,1
LogAnnualizedEligibleSpendEUR,0.05939475456670471,null,2
SupplierSpendSharePct,0.0592118714597827,null,3
AnnualizedEligibleSpendEUR,0.05862060018081585,null,4
Rolling3YAverageSpendEUR,0.057182493213695486,null,5
AnnualizedFullyReceivedPOItemCount,0.05530165338951205,null,6
AnnualizedInvoiceCount,0.03762939495718916,null,7
SupplierOTDPct,0.036332870301588896,null,8
LateFullyReceivedPct,0.03340145620494225,null,9
MaverickSpendPct,0.03256084805893547,null,10


**Retrain selected model on all labeled history**

After the model architecture has been validated on 2024, we use all labeled years:

- 2022
- 2023
- 2024

to create the production scoring candidate.

In [0]:
# ============================================================
# Retrain production candidate using all labeled history
# ============================================================

all_labeled_pd = (
    training_pd[
        training_pd[
            "FeatureYear"
        ].isin(
            [
                TRAIN_YEAR,
                VALIDATION_YEAR,
                TEST_YEAR
            ]
        )
    ]
    .copy()
)


X_all_labeled = (
    all_labeled_pd[
        model_features
    ].copy()
)


y_all_labeled = (
    all_labeled_pd[
        TARGET_COLUMN
    ]
    .astype(int)
)


production_pipeline = (
    fit_model_pipeline(
        BEST_MODEL_NAME,
        X_all_labeled,
        y_all_labeled
    )
)


print(
    "Production candidate trained."
)

print(
    "Labeled training rows:",
    f"{len(X_all_labeled):,}"
)

print(
    "Years:",
    sorted(
        all_labeled_pd[
            "FeatureYear"
        ]
        .unique()
        .tolist()
    )
)

Production candidate trained.
Labeled training rows: 1,009
Years: [2022, 2023, 2024]


**Log production scoring model**

In [0]:
# ============================================================
# Log production scoring candidate
# ============================================================

if mlflow.active_run() is not None:

    mlflow.end_run()


with mlflow.start_run(
    run_name=(
        f"supplier_risk_"
        f"{BEST_MODEL_NAME}_"
        f"production_candidate"
    )
) as production_run:

    mlflow.set_tags({
        "project":
            "Enterprise Procurement Intelligence Platform",

        "model_family":
            "Supplier Risk",

        "selected_model":
            BEST_MODEL_NAME,

        "model_stage":
            "production_candidate",

        "scoring_year":
            str(
                SCORING_YEAR
            )
    })


    mlflow.log_params({
        "training_years":
            (
                f"{TRAIN_YEAR},"
                f"{VALIDATION_YEAR},"
                f"{TEST_YEAR}"
            ),

        "decision_threshold":
            SELECTED_THRESHOLD,

        "threshold_objective":
            "F1",

        "threshold_source":
            (
                "2022-2023 grouped OOF "
                "F1 optimum"
            )
    })


    # --------------------------------------------------------
    # Reference metrics
    #
    # The production fit includes the former 2024 holdout.
    # Therefore these remain historical temporal-test metrics,
    # not new production-fit evaluation metrics.
    # --------------------------------------------------------

    mlflow.log_metrics({
        "reference_test_roc_auc":
            test_metrics[
                "roc_auc"
            ],

        "reference_test_pr_auc":
            test_metrics[
                "pr_auc"
            ],

        "reference_test_precision":
            test_metrics[
                "precision"
            ],

        "reference_test_recall":
            test_metrics[
                "recall"
            ],

        "reference_test_f1":
            test_metrics[
                "f1"
            ],

        "reference_test_f2":
            test_metrics[
                "f2"
            ]
    })


    production_model_info = (
        log_sklearn_model_compat(
            production_pipeline,
            X_all_labeled,
            model_name="model"
        )
    )


    PRODUCTION_RUN_ID = (
        production_run.info.run_id
    )


    PRODUCTION_MODEL_URI = (
        production_model_info.model_uri
    )


print(
    "Production candidate logged."
)

print(
    "Run ID:",
    PRODUCTION_RUN_ID
)

print(
    "Model URI:",
    PRODUCTION_MODEL_URI
)

print(
    "Production decision-threshold objective: F1"
)

🔗 View Logged Model at: https://adb-7405606393632123.3.azuredatabricks.net/ml/experiments/445824903590900/models/m-221908c9ba3a44f6a0f949889e2380ec?o=7405606393632123


Production candidate logged.
Run ID: b4e4b6964d364b01b117bf86a278f5eb
Model URI: models:/m-221908c9ba3a44f6a0f949889e2380ec
Production decision-threshold objective: F1


**Score the 2026 supplier population**

Creates the Supplier Risk Score 0–100.

In [0]:
# ============================================================
# Score current 2026 supplier population
# ============================================================

X_scoring = (
    scoring_pd[
        model_features
    ].copy()
)


scoring_probability = (
    production_pipeline
    .predict_proba(
        X_scoring
    )[:, 1]
)


scoring_predicted_flag = (
    scoring_probability
    >= SELECTED_THRESHOLD
).astype(int)


prediction_context_columns = [
    "SupplierKey",
    "SupplierID",
    "SupplierName",
    "FeatureYear",
    "SourceAsOfDate",

    "SupplierType",
    "Country",
    "Region",
    "ESGRating",
    "FinancialRiskScore",

    "SupplierOTDPct",
    "OverdueOpenDeliveryExposurePct",
    "InvoiceDisputePct",
    "SpendVolatilityPct",
    "AnnualizedEligibleSpendEUR",
    "RiskMetricCoveragePct"
]


supplier_risk_predictions_pd = (
    scoring_pd[
        prediction_context_columns
    ]
    .copy()
)


supplier_risk_predictions_pd[
    "SupplierRiskProbability"
] = scoring_probability


supplier_risk_predictions_pd[
    "SupplierRiskScore"
] = (
    scoring_probability
    * 100.0
)


supplier_risk_predictions_pd[
    "PredictedHighRiskFlag"
] = scoring_predicted_flag


supplier_risk_predictions_pd[
    "DecisionThreshold"
] = SELECTED_THRESHOLD


supplier_risk_predictions_pd[
    "ModelName"
] = BEST_MODEL_NAME


supplier_risk_predictions_pd[
    "ModelRunID"
] = PRODUCTION_RUN_ID


supplier_risk_predictions_pd[
    "ModelURI"
] = PRODUCTION_MODEL_URI


supplier_risk_predictions_pd[
    "PredictionTimestampUTC"
] = datetime.now(
    timezone.utc
)


supplier_risk_predictions_pd = (
    supplier_risk_predictions_pd

    .sort_values(
        "SupplierRiskScore",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


print(
    "2026 suppliers scored:",
    f"{len(supplier_risk_predictions_pd):,}"
)

2026 suppliers scored: 356


**Inspect highest-risk suppliers**

In [0]:
# ============================================================
# Inspect highest-risk suppliers
# ============================================================

display(
    supplier_risk_predictions_pd[
        [
            "SupplierID",
            "SupplierName",
            "SupplierRiskScore",
            "PredictedHighRiskFlag",

            "SupplierOTDPct",
            "OverdueOpenDeliveryExposurePct",
            "InvoiceDisputePct",
            "SpendVolatilityPct",

            "FinancialRiskScore",
            "ESGRating",

            "AnnualizedEligibleSpendEUR"
        ]
    ]
    .head(25)
)

SupplierID,SupplierName,SupplierRiskScore,PredictedHighRiskFlag,SupplierOTDPct,OverdueOpenDeliveryExposurePct,InvoiceDisputePct,SpendVolatilityPct,FinancialRiskScore,ESGRating,AnnualizedEligibleSpendEUR
SUP000464,"Meridian Advanced Solutions Co., Ltd.",73.71794886390036,1,80.46,23.684210526315788,20.0,66.66749525691291,68.0,C,2863809.300235849
SUP000191,Meridian Integrated Resources N.V.,71.67990250713547,1,76.25,20.266666666666666,7.142857142857142,109.9039469479689,84.0,D,8109081.054952829
SUP000102,Pioneer Dynamic Mechanical Ltd.,70.48807107500042,1,86.25,13.978494623655912,7.142857142857142,73.7733099891663,49.0,null,2912981.7731132074
SUP000124,Keystone Sustainable Systems Corp.,70.39585981736447,1,72.6,10.975609756097562,9.523809523809524,49.539315505919035,92.0,C,9050192.871226415
SUP000089,Pioneer Global Distribution Ltd.,70.19726811629954,1,81.9,16.666666666666664,17.073170731707318,58.19123167267887,51.0,B,3520559.671226415
SUP000365,Cobalt Precision Logistics PLC,69.86758084628867,1,78.95,20.833333333333336,8.695652173913043,43.202099117951505,70.0,C,2.730597423018868E7
SUP000066,Nordic International Equipment Ltd.,67.78798885159526,1,79.71,41.02564102564102,11.428571428571429,74.1511728923788,99.0,A,3611946.6667452827
SUP000258,Titan Integrated Automation N.V.,66.88225963676844,1,85.71,44.73684210526316,7.6923076923076925,119.0515959267071,97.0,B,2968685.197877358
SUP000178,Terra International Supply S.A.S.,66.830608018979,1,80.81,15.384615384615385,6.521739130434782,105.44819436143989,58.0,B,3629770.8735849056
SUP000259,Meridian Precision Resources Corp.,66.27313137271264,1,91.4,6.0606060606060606,8.823529411764707,15.591488897070743,50.0,A,9151998.189150942


**Inspect 2026 risk-score distribution**

In [0]:
# ============================================================
# Inspect current supplier risk distribution
# ============================================================

risk_distribution_pd = (
    supplier_risk_predictions_pd

    .groupby(
        "PredictedHighRiskFlag",
        dropna=False
    )

    .agg(
        SupplierCount=(
            "SupplierID",
            "count"
        ),

        AverageRiskScore=(
            "SupplierRiskScore",
            "mean"
        ),

        AverageAnnualizedSpendEUR=(
            "AnnualizedEligibleSpendEUR",
            "mean"
        )
    )

    .reset_index()
)


risk_distribution_pd[
    "SupplierPct"
] = (
    risk_distribution_pd[
        "SupplierCount"
    ]
    /
    len(
        supplier_risk_predictions_pd
    )
    * 100.0
)


display(
    risk_distribution_pd
)


print(
    "\nRisk score quantiles:"
)

print(
    supplier_risk_predictions_pd[
        "SupplierRiskScore"
    ]
    .quantile(
        [
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
)

PredictedHighRiskFlag,SupplierCount,AverageRiskScore,AverageAnnualizedSpendEUR,SupplierPct
0,152,35.534625070140564,6249041.719732188,42.69662921348314
1,204,51.469901778616595,1.4059183461509202E7,57.30337078651685



Risk score quantiles:
0.25    37.507074
0.50    43.565637
0.75    51.281502
0.90    57.453143
0.95    62.152761
Name: SupplierRiskScore, dtype: float64


**B_03 model quality gates**

This validates the technical validity of the model output.

In [0]:
# ============================================================
# DB_03 model quality gates
#
# Technical validity
# +
# minimum predictive-skill checks
# +
# anti-degeneracy checks
# ============================================================


# ------------------------------------------------------------
# Test-set diagnostics
# ------------------------------------------------------------

test_prevalence = float(
    y_test.mean()
)


test_predicted_high_risk_pct = float(
    np.mean(
        test_predictions
    )
)


test_predicted_class_count = int(
    np.unique(
        test_predictions
    ).size
)


# ------------------------------------------------------------
# Current scoring-population diagnostics
# ------------------------------------------------------------

scoring_count = len(
    supplier_risk_predictions_pd
)


scoring_unique_supplier_count = (
    supplier_risk_predictions_pd[
        "SupplierID"
    ]
    .nunique()
)


scoring_predicted_high_risk_pct = float(
    supplier_risk_predictions_pd[
        "PredictedHighRiskFlag"
    ]
    .mean()
)


scoring_predicted_class_count = int(
    supplier_risk_predictions_pd[
        "PredictedHighRiskFlag"
    ]
    .nunique()
)


# ------------------------------------------------------------
# Risk-score validity
# ------------------------------------------------------------

null_risk_score_count = int(
    supplier_risk_predictions_pd[
        "SupplierRiskScore"
    ]
    .isna()
    .sum()
)


invalid_risk_score_count = int(
    (
        (
            supplier_risk_predictions_pd[
                "SupplierRiskScore"
            ]
            < 0
        )
        |
        (
            supplier_risk_predictions_pd[
                "SupplierRiskScore"
            ]
            > 100
        )
    )
    .sum()
)


# ============================================================
# Quality checks
# ============================================================

quality_checks = [

    (
        "Selected model exists",

        BEST_MODEL_NAME
        in candidate_specs
    ),


    (
        "Decision threshold is valid",

        (
            SELECTED_THRESHOLD
            > 0

            and

            SELECTED_THRESHOLD
            < 1
        )
    ),


    (
        "Temporal test contains both actual classes",

        y_test.nunique()
        == 2
    ),


    (
        "Temporal test predicts both classes",

        test_predicted_class_count
        == 2
    ),


    (
        "2026 scoring population predicts both classes",

        scoring_predicted_class_count
        == 2
    ),


    (
        "Test ROC-AUC is finite",

        math.isfinite(
            test_metrics[
                "roc_auc"
            ]
        )
    ),


    (
        "Test ROC-AUC exceeds random ranking",

        test_metrics[
            "roc_auc"
        ]
        > 0.50
    ),


    (
        "Test PR-AUC is finite",

        math.isfinite(
            test_metrics[
                "pr_auc"
            ]
        )
    ),


    (
        "Test PR-AUC exceeds prevalence baseline",

        test_metrics[
            "pr_auc"
        ]
        > test_prevalence
    ),


    (
        "Scoring population contains rows",

        scoring_count
        > 0
    ),


    (
        "Scoring grain is unique by supplier",

        scoring_count
        ==
        scoring_unique_supplier_count
    ),


    (
        "Supplier Risk Score contains no nulls",

        null_risk_score_count
        == 0
    ),


    (
        "Supplier Risk Score is between 0 and 100",

        invalid_risk_score_count
        == 0
    )
]


# ============================================================
# Execute quality checks
# ============================================================

failed_checks = []


for (
    check_name,
    passed
) in quality_checks:

    print(
        f"{'PASS' if passed else 'FAIL'} | "
        f"{check_name}"
    )


    if not passed:

        failed_checks.append(
            check_name
        )


# ============================================================
# Model-quality diagnostics
# ============================================================

print(
    "\nMODEL QUALITY DIAGNOSTICS"
)


print(
    "2024 actual high-risk prevalence:",
    f"{test_prevalence:.2%}"
)


print(
    "2024 predicted high-risk share:",
    f"{test_predicted_high_risk_pct:.2%}"
)


print(
    "2026 predicted high-risk share:",
    f"{scoring_predicted_high_risk_pct:.2%}"
)


print(
    "\n2024 test ROC-AUC:",
    round(
        test_metrics[
            "roc_auc"
        ],
        4
    )
)


print(
    "2024 test PR-AUC:",
    round(
        test_metrics[
            "pr_auc"
        ],
        4
    )
)


print(
    "2024 prevalence PR baseline:",
    round(
        test_prevalence,
        4
    )
)


print(
    "2024 test precision:",
    round(
        test_metrics[
            "precision"
        ],
        4
    )
)


print(
    "2024 test recall:",
    round(
        test_metrics[
            "recall"
        ],
        4
    )
)


print(
    "2024 test F1:",
    round(
        test_metrics[
            "f1"
        ],
        4
    )
)


print(
    "Selected threshold:",
    round(
        SELECTED_THRESHOLD,
        4
    )
)


print(
    "Threshold objective:",
    "F1"
)


# ------------------------------------------------------------
# BRD precision target remains a reported target only
# ------------------------------------------------------------

print(
    "\nBRD precision target:",
    f"{BRD_PRECISION_TARGET:.0%}"
)


print(
    "BRD precision target met:",
    (
        "YES"
        if test_metrics[
            "precision"
        ]
        >= BRD_PRECISION_TARGET
        else "NO"
    )
)


# ============================================================
# Final gate
# ============================================================

if failed_checks:

    raise ValueError(
        "DB_03 quality gate FAILED: "
        + "; ".join(
            failed_checks
        )
    )


print(
    "\nDB_03 MODEL QUALITY GATE PASSED."
)

PASS | Selected model exists
PASS | Decision threshold is valid
PASS | Temporal test contains both actual classes
PASS | Temporal test predicts both classes
PASS | 2026 scoring population predicts both classes
PASS | Test ROC-AUC is finite
PASS | Test ROC-AUC exceeds random ranking
PASS | Test PR-AUC is finite
PASS | Test PR-AUC exceeds prevalence baseline
PASS | Scoring population contains rows
PASS | Scoring grain is unique by supplier
PASS | Supplier Risk Score contains no nulls
PASS | Supplier Risk Score is between 0 and 100

MODEL QUALITY DIAGNOSTICS
2024 actual high-risk prevalence: 27.83%
2024 predicted high-risk share: 44.35%
2026 predicted high-risk share: 57.30%

2024 test ROC-AUC: 0.5494
2024 test PR-AUC: 0.3588
2024 prevalence PR baseline: 0.2783
2024 test precision: 0.3007
2024 test recall: 0.4792
2024 test F1: 0.3695
Selected threshold: 0.418
Threshold objective: F1

BRD precision target: 85%
BRD precision target met: NO

DB_03 MODEL QUALITY GATE PASSED.


Build model metadata

In [0]:
# ============================================================
# Build supplier-risk model metadata
# ============================================================

selected_development_row = (
    candidate_results_pd[
        candidate_results_pd[
            "ModelName"
        ]
        == BEST_MODEL_NAME
    ]
    .iloc[0]
)


model_metadata_pd = pd.DataFrame(
    [
        {
            "ModelFamily":
                "Supplier Risk",

            "ModelName":
                BEST_MODEL_NAME,

            "ModelRunID":
                PRODUCTION_RUN_ID,

            "ModelURI":
                PRODUCTION_MODEL_URI,

            "DevelopmentStartYear":
                TRAIN_YEAR,

            "DevelopmentEndYear":
                VALIDATION_YEAR,

            "DevelopmentMethod":
                (
                    "StratifiedGroupKFold "
                    "by SupplierID"
                ),

            "DevelopmentCVFolds":
                CV_FOLDS,

            "TemporalTestYear":
                TEST_YEAR,

            "ProductionTrainingEndYear":
                TEST_YEAR,

            "ScoringYear":
                SCORING_YEAR,

            "DecisionThreshold":
                SELECTED_THRESHOLD,

            "ThresholdObjective":
                "F1",

            "DevelopmentOOFROCAUC":
                float(
                    selected_development_row[
                        "DevelopmentOOFROCAUC"
                    ]
                ),

            "DevelopmentOOFPRAUC":
                float(
                    selected_development_row[
                        "DevelopmentOOFPRAUC"
                    ]
                ),

            "DevelopmentOOFPrecision":
                float(
                    selected_development_row[
                        "DevelopmentOOFPrecision"
                    ]
                ),

            "DevelopmentOOFRecall":
                float(
                    selected_development_row[
                        "DevelopmentOOFRecall"
                    ]
                ),

            "DevelopmentOOFF1":
                float(
                    selected_development_row[
                        "DevelopmentOOFF1"
                    ]
                ),

            "TestROCAUC":
                test_metrics[
                    "roc_auc"
                ],

            "TestPRAUC":
                test_metrics[
                    "pr_auc"
                ],

            "TestPrecision":
                test_metrics[
                    "precision"
                ],

            "TestRecall":
                test_metrics[
                    "recall"
                ],

            "TestF1":
                test_metrics[
                    "f1"
                ],

            "TestF2":
                test_metrics[
                    "f2"
                ],

            "TestActualHighRiskPct":
                test_prevalence
                * 100.0,

            "TestPredictedHighRiskPct":
                test_predicted_high_risk_pct
                * 100.0,

            "ScoringPredictedHighRiskPct":
                scoring_predicted_high_risk_pct
                * 100.0,

            "BRDPrecisionTarget":
                BRD_PRECISION_TARGET,

            "BRDPrecisionTargetMetFlag":
                int(
                    test_metrics[
                        "precision"
                    ]
                    >= BRD_PRECISION_TARGET
                ),

            "TrainingRowCount":
                int(
                    len(
                        X_all_labeled
                    )
                ),

            "ScoringRowCount":
                int(
                    scoring_count
                ),

            "CreatedTimestampUTC":
                datetime.now(
                    timezone.utc
                )
        }
    ]
)


display(
    model_metadata_pd
)

ModelFamily,ModelName,ModelRunID,ModelURI,DevelopmentStartYear,DevelopmentEndYear,DevelopmentMethod,DevelopmentCVFolds,TemporalTestYear,ProductionTrainingEndYear,ScoringYear,DecisionThreshold,ThresholdObjective,DevelopmentOOFROCAUC,DevelopmentOOFPRAUC,DevelopmentOOFPrecision,DevelopmentOOFRecall,DevelopmentOOFF1,TestROCAUC,TestPRAUC,TestPrecision,TestRecall,TestF1,TestF2,TestActualHighRiskPct,TestPredictedHighRiskPct,ScoringPredictedHighRiskPct,BRDPrecisionTarget,BRDPrecisionTargetMetFlag,TrainingRowCount,ScoringRowCount,CreatedTimestampUTC
Supplier Risk,RandomForest,b4e4b6964d364b01b117bf86a278f5eb,models:/m-221908c9ba3a44f6a0f949889e2380ec,2022,2023,StratifiedGroupKFold by SupplierID,5,2024,2024,2026,0.4179641151487885,F1,0.6677537662923886,0.4265984961623047,0.40397350993377484,0.6594594594594595,0.5010266940451745,0.5494059571619814,0.35876460738766247,0.3006535947712418,0.4791666666666667,0.36947791164658633,0.42830540037243947,27.82608695652174,44.34782608695652,57.30337078651685,0.85,0,1009,356,2026-08-13T04:46:04.621039Z


**Persist supplier-risk outputs to OneLake**

These remain ML intermediate data products under /Files.

In [0]:
# ============================================================
# Persist DB_03 ML outputs to OneLake
# ============================================================

supplier_risk_predictions_spark_df = (
    spark.createDataFrame(
        supplier_risk_predictions_pd
    )
)


model_metadata_spark_df = (
    spark.createDataFrame(
        model_metadata_pd
    )
)


feature_importance_output_pd = (
    feature_importance_pd
    .copy()
)


feature_importance_output_pd[
    "ModelName"
] = BEST_MODEL_NAME


feature_importance_output_pd[
    "ModelRunID"
] = PRODUCTION_RUN_ID


feature_importance_output_pd[
    "CreatedTimestampUTC"
] = datetime.now(
    timezone.utc
)


feature_importance_spark_df = (
    spark.createDataFrame(
        feature_importance_output_pd
    )
)


# ------------------------------------------------------------
# Predictions
# ------------------------------------------------------------

(
    supplier_risk_predictions_spark_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .save(
        PREDICTIONS_PATH
    )
)


# ------------------------------------------------------------
# Model metadata
# ------------------------------------------------------------

(
    model_metadata_spark_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .save(
        MODEL_METADATA_PATH
    )
)


# ------------------------------------------------------------
# Feature importance
# ------------------------------------------------------------

(
    feature_importance_spark_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .save(
        FEATURE_IMPORTANCE_PATH
    )
)


print(
    "DB_03 outputs written successfully."
)

DB_03 outputs written successfully.


**Final persistance validation**

In [0]:
# ============================================================
# Validate DB_03 persisted outputs
# ============================================================

prediction_validation_df = (
    spark.read
    .format("delta")
    .load(
        PREDICTIONS_PATH
    )
)


metadata_validation_df = (
    spark.read
    .format("delta")
    .load(
        MODEL_METADATA_PATH
    )
)


feature_importance_validation_df = (
    spark.read
    .format("delta")
    .load(
        FEATURE_IMPORTANCE_PATH
    )
)


persisted_prediction_count = (
    prediction_validation_df.count()
)


persisted_metadata_count = (
    metadata_validation_df.count()
)


persisted_feature_importance_count = (
    feature_importance_validation_df.count()
)


if (
    persisted_prediction_count
    != scoring_count
):

    raise ValueError(
        "Supplier-risk prediction persistence "
        "validation failed."
    )


if (
    persisted_metadata_count
    != 1
):

    raise ValueError(
        "Supplier-risk model metadata "
        "validation failed."
    )


if (
    persisted_feature_importance_count
    <= 0
):

    raise ValueError(
        "Supplier-risk feature importance "
        "validation failed."
    )


print(
    "DB_03 persistence validation PASSED."
)

print(
    "Predictions:",
    f"{persisted_prediction_count:,}"
)

print(
    "Model metadata rows:",
    persisted_metadata_count
)

print(
    "Feature importance rows:",
    persisted_feature_importance_count
)

print(
    "\nDB_03 SUPPLIER RISK MODEL PASSED."
)

DB_03 persistence validation PASSED.
Predictions: 356
Model metadata rows: 1
Feature importance rows: 88

DB_03 SUPPLIER RISK MODEL PASSED.
